# LTX-Video API + Product Video Generator
Setup + Run

In [ ]:
!git clone https://github.com/Saqo12/ltx-video-api.git
%cd ltx-video-api

In [ ]:
!pip install -q transformers torch peft accelerate bitsandbytes
!pip install -q python-multipart aiohttp httpx

In [ ]:
!pip install -q ltx-video
!pip install -q pyngrok
!ngrok config add-authtoken 2FputHwfFi1KyGWRjkJfFWP7YqGA_7KiNMqG3mXnNFRUC7X5c

In [ ]:
# STEP 1: Get HF token at https://huggingface.co/settings/tokens
# STEP 2: Accept license at https://huggingface.co/Lightricks/LTX-Video
HF_TOKEN = input("Paste your HuggingFace token: ").strip()
!huggingface-cli login --token $HF_TOKEN

In [ ]:
from ltx_video import LTXVideoPipeline
pipe = LTXVideoPipeline.from_pretrained("Lightricks/LTX-Video",
                                       torch_dtype=torch.bfloat16)
pipe.to("cuda")

In [ ]:
from ngrok import ngrok
tunnel = ngrok.connect(8000)
print("PUBLIC URL:", tunnel.public_url)

In [ ]:
from fastapi import FastAPI
import uvicorn
app = FastAPI()

@app.get("/health")
def health(): return {"status": "ok", "model": "LTX-Video"}

@app.post("/generate")
async def generate(req: dict):
    video = pipe.generate(
        prompt=req["prompt"],
        num_inference_steps=req.get("steps", 40),
        guidance_scale=req.get("guidance", 4.0),
        num_videos=1,
        height=512, width=512,
        duration=req.get("duration", 5)
    ).video[0]
    out_path = "/tmp/output.mp4"
    video.save(out_path)
    return {"url": out_path}

uvicorn.run(app, port=8000, host="0.0.0.0")